# Pipeline de Minería de Datos (E-commerce) — Predicción de compra + Clustering

Este notebook implementa un pipeline completo:
1. **Generación** de datos ficticios (volumen mediano).
2. **Predicción de compra** (clasificación) con *scikit-learn* (preprocesado + modelo).
3. **Clustering** (segmentación de clientes) con K-Means.
4. **Exportación** de artefactos a `../artifacts`.

> Dataset simulado: sesiones de navegación + atributos de cliente + contexto de marketing.


In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

RNG = np.random.default_rng(42)

ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)


## 1) Generación de datos ficticios (volumen mediano)

In [2]:
def generate_customers(n_customers: int = 10000, rng=RNG) -> pd.DataFrame:
    customer_id = np.arange(1, n_customers + 1)

    regions = np.array(["Norte", "Centro", "Sur", "Islas"])
    devices_pref = np.array(["Desktop", "Mobile", "Tablet"])
    acquisition = np.array(["SEO", "SEM", "Email", "Social", "Afiliados", "Directo"])

    df = pd.DataFrame({
        "customer_id": customer_id,
        "age": rng.integers(18, 70, size=n_customers),
        "tenure_months": rng.integers(0, 72, size=n_customers),
        "region": rng.choice(regions, size=n_customers, p=[0.25, 0.40, 0.25, 0.10]),
        "preferred_device": rng.choice(devices_pref, size=n_customers, p=[0.35, 0.55, 0.10]),
        "acquisition_channel": rng.choice(acquisition, size=n_customers, p=[0.25, 0.20, 0.15, 0.15, 0.10, 0.15]),
        "loyalty_member": rng.choice([0, 1], size=n_customers, p=[0.65, 0.35]),
    })

    # Score socioeconómico sintético (0-100) correlacionado con edad y región (ligero)
    region_shift = df["region"].map({"Norte": 3, "Centro": 1, "Sur": -2, "Islas": 2}).astype(float)
    df["socio_score"] = np.clip(
        50 + 0.2*(df["age"] - 35) + region_shift*2 + rng.normal(0, 12, size=n_customers),
        0, 100
    )

    return df

def generate_sessions(customers: pd.DataFrame, n_sessions: int = 50000, rng=RNG) -> pd.DataFrame:
    customer_ids = rng.choice(customers["customer_id"].values, size=n_sessions, replace=True)

    traffic = np.array(["Organic", "Paid", "Email", "Social", "Referral", "Direct"])
    campaign = np.array(["None", "Promo10", "Promo20", "FreeShip", "Bundle", "Flash"])

    # Categorías vistas (la sesión puede centrarse en una principal)
    category = np.array(["Electrónica", "Hogar", "Moda", "Deporte", "Libros", "Juguetes"])

    df = pd.DataFrame({
        "session_id": np.arange(1, n_sessions + 1),
        "customer_id": customer_ids,
        "traffic_source": rng.choice(traffic, size=n_sessions, p=[0.35, 0.25, 0.10, 0.10, 0.05, 0.15]),
        "campaign": rng.choice(campaign, size=n_sessions, p=[0.55, 0.10, 0.08, 0.10, 0.10, 0.07]),
        "category_main": rng.choice(category, size=n_sessions),
        "hour": rng.integers(0, 24, size=n_sessions),
        "day_of_week": rng.integers(0, 7, size=n_sessions),  # 0=lunes
    })

    # Métricas de comportamiento (distribuciones razonables)
    df["pages_viewed"] = np.clip(rng.poisson(6, size=n_sessions) + 1, 1, 60)
    df["time_on_site_sec"] = np.clip((rng.gamma(2.0, 90.0, size=n_sessions)).astype(int), 10, 7200)
    df["items_viewed"] = np.clip(rng.poisson(3, size=n_sessions), 0, 80)

    # Señales del embudo
    df["added_to_cart"] = (rng.random(n_sessions) < (0.12 + 0.01*np.log1p(df["pages_viewed"]))).astype(int)
    df["checkout_started"] = ((df["added_to_cart"] == 1) & (rng.random(n_sessions) < 0.45)).astype(int)

    # Precio medio del carrito (si aplica)
    base_price = rng.normal(45, 20, size=n_sessions)
    cat_mult = df["category_main"].map({
        "Electrónica": 1.8, "Hogar": 1.2, "Moda": 1.1, "Deporte": 1.3, "Libros": 0.7, "Juguetes": 0.9
    }).astype(float).values
    df["avg_item_price"] = np.clip(base_price * cat_mult, 5, 400)

    # Descuento efectivo (según campaña)
    disc = df["campaign"].map({
        "None": 0.00, "Promo10": 0.10, "Promo20": 0.20, "FreeShip": 0.05, "Bundle": 0.12, "Flash": 0.18
    }).astype(float)
    df["discount_rate"] = disc

    # Unimos atributos de cliente para construir la probabilidad de compra (label realista)
    c = customers.set_index("customer_id").loc[df["customer_id"]].reset_index(drop=True)

    # Efectos: comportamiento + lealtad + descuento + canal + precio -> compra
    # Construimos log-odds y luego muestreamos Bernoulli.
    traffic_bonus = df["traffic_source"].map({
        "Organic": 0.10, "Paid": 0.05, "Email": 0.20, "Social": -0.05, "Referral": 0.08, "Direct": 0.12
    }).astype(float).values

    device_bonus = c["preferred_device"].map({"Desktop": 0.10, "Mobile": -0.02, "Tablet": 0.00}).astype(float).values
    loyalty_bonus = (c["loyalty_member"] * 0.25).astype(float).values

    engagement = 0.015*np.log1p(df["pages_viewed"]) + 0.00025*np.log1p(df["time_on_site_sec"]) + 0.02*df["added_to_cart"] + 0.35*df["checkout_started"]
    price_penalty = -0.0012*(df["avg_item_price"] - 40)  # precio alto reduce conversión
    discount_bonus = 1.4*df["discount_rate"].values       # descuento incrementa conversión

    tenure_effect = 0.004*np.clip(c["tenure_months"].values, 0, 72)  # clientes antiguos convierten algo más
    socio_effect = 0.003*(c["socio_score"].values - 50)

    log_odds = -3.2 + engagement + price_penalty + discount_bonus + traffic_bonus + device_bonus + loyalty_bonus + tenure_effect + socio_effect
    p_purchase = 1/(1 + np.exp(-log_odds))

    df["purchase"] = (rng.random(n_sessions) < p_purchase).astype(int)

    # Cantidad comprada (si hay compra)
    df["items_purchased"] = 0
    bought_idx = df["purchase"] == 1
    df.loc[bought_idx, "items_purchased"] = np.clip(rng.poisson(2.2, bought_idx.sum()) + 1, 1, 20)

    # Ticket medio (si hay compra)
    df["order_value"] = 0.0
    df.loc[bought_idx, "order_value"] = (df.loc[bought_idx, "items_purchased"] * df.loc[bought_idx, "avg_item_price"] * (1 - df.loc[bought_idx, "discount_rate"])).round(2)

    return df

customers = generate_customers(n_customers=10000)
sessions = generate_sessions(customers, n_sessions=50000)

customers.head(), sessions.head(), sessions["purchase"].mean()


(   customer_id  age  tenure_months  region preferred_device  \
 0            1   22             24     Sur           Mobile   
 1            2   58             15     Sur           Tablet   
 2            3   52             37   Norte          Desktop   
 3            4   40             47   Norte          Desktop   
 4            5   40             18  Centro           Mobile   
 
   acquisition_channel  loyalty_member  socio_score  
 0                 SEM               0    43.976528  
 1                 SEM               0    40.359575  
 2             Directo               0    64.909932  
 3                 SEO               0    59.044095  
 4                 SEO               1    50.376823  ,
    session_id  customer_id traffic_source  campaign category_main  hour  \
 0           1         2187         Social    Bundle          Moda    13   
 1           2         4093        Organic      None       Deporte     3   
 2           3         9504        Organic  FreeShip         

### Vista rápida del dataset

In [3]:
sessions.describe(include="all").T.head(20)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
session_id,50000.0,NaN,NaN,NaN,25000.5,14433.901067,1.0,12500.75,25000.5,37500.25,50000.0
customer_id,50000.0,NaN,NaN,NaN,5008.62578,2874.144428,1.0,2527.0,5004.0,7480.0,10000.0
traffic_source,50000,6,Organic,17462,NaN,NaN,NaN,NaN,NaN,NaN,NaN
campaign,50000,6,None,27498,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category_main,50000,6,Deporte,8441,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hour,50000.0,NaN,NaN,NaN,11.53414,6.924296,0.0,6.0,12.0,18.0,23.0
day_of_week,50000.0,NaN,NaN,NaN,2.98704,2.003365,0.0,1.0,3.0,5.0,6.0
pages_viewed,50000.0,NaN,NaN,NaN,6.99686,2.442443,1.0,5.0,7.0,9.0,21.0
time_on_site_sec,50000.0,NaN,NaN,NaN,179.54324,127.709129,10.0,86.0,151.0,242.0,1470.0
items_viewed,50000.0,NaN,NaN,NaN,3.0085,1.743941,0.0,2.0,3.0,4.0,13.0


## 2) Predicción de compra (clasificación)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib

# Features: combinamos sesión + algunos atributos cliente (join)
df = sessions.merge(customers, on="customer_id", how="left")

target = "purchase"

feature_cols = [
    "traffic_source", "campaign", "category_main",
    "hour", "day_of_week",
    "pages_viewed", "time_on_site_sec", "items_viewed",
    "added_to_cart", "checkout_started",
    "avg_item_price", "discount_rate",
    "age", "tenure_months", "region", "preferred_device", "acquisition_channel", "loyalty_member", "socio_score"
]

X = df[feature_cols].copy()
y = df[target].astype(int).copy()

cat_cols = ["traffic_source", "campaign", "category_main", "region", "preferred_device", "acquisition_channel"]
num_cols = [c for c in feature_cols if c not in cat_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[("scaler", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

# Modelo baseline: regresión logística
logreg = Pipeline(steps=[
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=2000, n_jobs=None))
])

logreg.fit(X_train, y_train)

proba = logreg.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("AUC:", roc_auc_score(y_test, proba))
print(classification_report(y_test, pred))
confusion_matrix(y_test, pred)


AUC: 0.5424863936086247
              precision    recall  f1-score   support

           0       0.94      1.00      0.97     11701
           1       0.00      0.00      0.00       799

    accuracy                           0.94     12500
   macro avg       0.47      0.50      0.48     12500
weighted avg       0.88      0.94      0.91     12500



/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


array([[11701,     0],
       [  799,     0]])

### Modelo alternativo: Random Forest (suele capturar no linealidades)

In [5]:
rf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    ))
])

rf.fit(X_train, y_train)

proba_rf = rf.predict_proba(X_test)[:, 1]
pred_rf = (proba_rf >= 0.5).astype(int)

print("AUC:", roc_auc_score(y_test, proba_rf))
print(classification_report(y_test, pred_rf))
confusion_matrix(y_test, pred_rf)


AUC: 0.5141196493908129
              precision    recall  f1-score   support

           0       0.94      1.00      0.97     11701
           1       0.00      0.00      0.00       799

    accuracy                           0.94     12500
   macro avg       0.47      0.50      0.48     12500
weighted avg       0.88      0.94      0.91     12500



/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


array([[11701,     0],
       [  799,     0]])

### Guardar artefactos del modelo

In [6]:
# Guardamos el mejor por AUC (simplemente comparamos aquí)
auc_lr = roc_auc_score(y_test, proba)
auc_rf = roc_auc_score(y_test, proba_rf)

best = rf if auc_rf >= auc_lr else logreg
best_name = "random_forest" if auc_rf >= auc_lr else "logistic_regression"

joblib.dump(best, ARTIFACTS_DIR / f"model_{best_name}.joblib")

metrics = pd.DataFrame([{
    "model": "logistic_regression",
    "auc": float(auc_lr),
    "purchase_rate_test": float(y_test.mean())
},{
    "model": "random_forest",
    "auc": float(auc_rf),
    "purchase_rate_test": float(y_test.mean())
}])

metrics.to_csv(ARTIFACTS_DIR / "metrics_models.csv", index=False)

metrics


,model,auc,purchase_rate_test
0,logistic_regression,0.542486,0.06392
1,random_forest,0.514120,0.06392


## 3) Clustering (segmentación de clientes)

In [7]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Agregamos sesiones a nivel cliente (features de comportamiento)
agg = df.groupby("customer_id").agg(
    sessions=("session_id", "count"),
    purchase_rate=("purchase", "mean"),
    avg_pages=("pages_viewed", "mean"),
    avg_time=("time_on_site_sec", "mean"),
    avg_items_viewed=("items_viewed", "mean"),
    cart_rate=("added_to_cart", "mean"),
    checkout_rate=("checkout_started", "mean"),
    avg_order_value=("order_value", "mean"),
    total_revenue=("order_value", "sum"),
).reset_index()

cust = customers.merge(agg, on="customer_id", how="left").fillna(0)

# Selección de variables para clustering
cluster_features = [
    "age", "tenure_months", "socio_score", "loyalty_member",
    "sessions", "purchase_rate", "avg_pages", "avg_time", "avg_items_viewed",
    "cart_rate", "checkout_rate", "avg_order_value", "total_revenue",
    "region", "preferred_device", "acquisition_channel"
]

Xc = cust[cluster_features].copy()

cat_cols_c = ["region", "preferred_device", "acquisition_channel"]
num_cols_c = [c for c in cluster_features if c not in cat_cols_c]

prep_c = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[("scaler", StandardScaler())]), num_cols_c),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_c),
    ]
)

Xc_mat = prep_c.fit_transform(Xc)

# Elegimos K con una búsqueda simple usando silhouette (k=3..8)
scores = []
models = []
for k in range(3, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(Xc_mat)
    s = silhouette_score(Xc_mat, labels)
    scores.append((k, float(s)))
    models.append((k, km))

scores_df = pd.DataFrame(scores, columns=["k", "silhouette"])
scores_df


,k,silhouette
0,3,0.106611
1,4,0.092177
2,5,0.073150
3,6,0.071788
4,7,0.071562
5,8,0.069730


In [8]:
best_k = scores_df.sort_values("silhouette", ascending=False).iloc[0]["k"]
best_k = int(best_k)

km_best = dict(models)[best_k]
cust["cluster"] = km_best.predict(Xc_mat)

# Guardamos el preprocesador y el modelo de clustering
joblib.dump(prep_c, ARTIFACTS_DIR / "cluster_preprocess.joblib")
joblib.dump(km_best, ARTIFACTS_DIR / f"kmeans_k{best_k}.joblib")

cust[["customer_id", "cluster"]].to_csv(ARTIFACTS_DIR / "customer_clusters.csv", index=False)

best_k, cust["cluster"].value_counts().sort_index()


(3,
 cluster
 0    2458
 1    5950
 2    1592
 Name: count, dtype: int64)

### Interpretación rápida de clusters

In [9]:
profile = cust.groupby("cluster").agg(
    n=("customer_id", "count"),
    tenure=("tenure_months", "mean"),
    loyalty=("loyalty_member", "mean"),
    sessions=("sessions", "mean"),
    purchase_rate=("purchase_rate", "mean"),
    revenue=("total_revenue", "mean"),
    avg_order=("avg_order_value", "mean"),
    socio=("socio_score", "mean"),
).round(3).sort_values("revenue", ascending=False)

profile


,n,tenure,loyalty,sessions,purchase_rate,revenue,avg_order,socio
cluster,,,,,,,,
2,1592,37.823,0.418,5.508,0.273,258.746,52.940,53.946
0,2458,34.664,0.344,5.381,0.034,16.827,2.800,53.549
1,5950,34.819,0.334,4.707,0.020,7.945,1.305,52.887


## 4) Ejemplo de uso: scoring de nuevas sesiones

In [10]:
# Cargamos el mejor modelo guardado y hacemos scoring a 5 sesiones de ejemplo
import joblib

# Reutilizamos 'best' del entrenamiento, pero mostramos el flujo como si fuese 'producción'
chosen_model_path = next(ARTIFACTS_DIR.glob("model_*.joblib"))
model = joblib.load(chosen_model_path)

sample = X_test.sample(5, random_state=7).copy()
score = model.predict_proba(sample)[:, 1]

out = sample.copy()
out["p_purchase"] = np.round(score, 4)
out.sort_values("p_purchase", ascending=False)


,traffic_source,campaign,category_main,hour,day_of_week,pages_viewed,time_on_site_sec,items_viewed,added_to_cart,checkout_started,avg_item_price,discount_rate,age,tenure_months,region,preferred_device,acquisition_channel,loyalty_member,socio_score,p_purchase
28064,Organic,Flash,Moda,16,1,8,68,4,0,0,40.376333,0.18,60,38,Sur,Mobile,Social,1,47.046650,0.0786
45562,Social,Bundle,Electrónica,14,2,6,391,2,1,1,67.089037,0.12,31,35,Centro,Mobile,Directo,0,46.190635,0.0679
2491,Paid,None,Moda,23,5,7,111,1,0,0,59.509770,0.00,39,36,Centro,Desktop,SEM,0,63.392039,0.0585
6011,Paid,Promo20,Electrónica,16,1,6,342,3,0,0,39.219785,0.20,27,48,Islas,Mobile,SEM,0,67.675511,0.0581
15734,Paid,None,Hogar,3,5,6,278,1,0,0,72.610103,0.00,41,40,Islas,Desktop,SEM,0,68.473515,0.0559
